# Feature Engineering

This notebook focuses on feature creation, transformation, and selection
based on insights from the exploratory data analysis.

In [1]:
import numpy as np
import pandas as pd

In [2]:
dataset = pd.read_excel("Dataset_Productivity.xlsx")

dataset['wip'] = dataset['wip'].fillna(0)
dataset['Dipartimento'] = dataset['Dipartimento'].str.strip().str.lower()

In [3]:
Q1 = dataset['DELTA'].quantile(0.25)
Q3 = dataset['DELTA'].quantile(0.75)
IQR = Q3 - Q1

limite_inferiore = Q1 - 1.5 * IQR
limite_superiore = Q3 + 1.5 * IQR

def classify_delta(delta):
    if delta < limite_inferiore:
        return 'Sovrastima'
    elif delta > limite_superiore:
        return 'Sottostima'
    else:
        return 'Stima realistica'

dataset['Delta_class'] = dataset['DELTA'].apply(classify_delta)

## Binary features
We create binary indicators to capture calendar effects.

In [4]:
dataset['FineMese'] = dataset['Quarto'].apply(lambda x: 1 if x == 'Quarter5' else 0)
dataset['Sabato'] = dataset['Giorno'].apply(lambda x: 1 if x == 'Saturday' else 0)

## Discretization of numerical variables
Numerical features are discretized to capture non-linear effects
and improve interpretability.

In [5]:
dataset['smv_bin'] = pd.qcut(
    dataset['smv'], 
    q=4, 
    labels=['Basso', 'Medio-Basso', 'Medio-Alto', 'Alto']
)

bins_wip = [0, 1, 1000, 5000, dataset['wip'].max()]
labels_wip = ['Zero', 'Basso', 'Medio', 'Alto']

dataset['wip_bin'] = pd.cut(
    dataset['wip'], 
    bins=bins_wip, 
    labels=labels_wip, 
    include_lowest=True
)

In [6]:
bins_incentivi = [0, 1, 100, 500, dataset['Incentivi'].max()]
labels_incentivi = ['Zero', 'Basso', 'Medio', 'Alto']

dataset['Incentivi_bin'] = pd.cut(
    dataset['Incentivi'],
    bins=bins_incentivi,
    labels=labels_incentivi,
    include_lowest=True
)

bins_orestraord = [0, 2, 4, dataset['MediaOreStraordinari'].max()]
labels_orestraord = ['Basso', 'Medio', 'Alto']

dataset['MediaOreStraordinari_bin'] = pd.cut(
    dataset['MediaOreStraordinari'],
    bins=bins_orestraord,
    labels=labels_orestraord,
    include_lowest=True
)

In [7]:
bins_lavoratori = [0, 40, 60, dataset['NumeroLavoratori'].max()]
labels_lavoratori = ['Piccolo', 'Medio', 'Grande']

dataset['NumeroLavoratori_bin'] = pd.cut(
    dataset['NumeroLavoratori'],
    bins=bins_lavoratori,
    labels=labels_lavoratori,
    include_lowest=True
)

dataset['CambioMansione'] = dataset['CambioMansione'].astype('category')

## Final feature set

We retain only engineered and interpretable features for modeling.

In [8]:
df_model = dataset[
    [
        'Delta_class',
        'Incentivi_bin',
        'wip_bin',
        'smv_bin',
        'NumeroLavoratori_bin',
        'Dipartimento',
        'FineMese',
        'Sabato'
    ]
]

df_model.head()

df_model.to_csv("dataset_model_ready.csv", index=False)

## Summary
This notebook produced a clean and interpretable feature set
ready for encoding, resampling, and machine learning modeling.